In [13]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cluster import KMeans
import scanpy as sc
import torch
import torch.optim as optim
import muon as mu
import yaml
import os


from scduo.scduo_perturbation.vae.data.data_loader import RNAseqLoader
from scduo.scduo_perturbation.vae.models.base.vae_model import EncoderModel

import argparse
from scduo.scduo_perturbation.diffusion.multimodal_script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    add_dict_to_argparser,
    args_to_dict
)
from scduo.scduo_perturbation.diffusion import dist_util

In [14]:
encoder_config = "/home/wuboyang/scduo-new/script/training_vae/configs/encoder/default.yaml"
dataset_path = '/home/wuboyang/scduo-new/dataset/processed_data/train_pbmc.h5ad'
covariate_keys = "cell_type" 
num_class = 7
ae_path = "/home/wuboyang/scduo-new/script/training_vae/outputs/checkpoints/my_vae/checkpoints/last-v2.ckpt"
adata = sc.read_h5ad(dataset_path)

In [15]:
# load autoencoder
with open(encoder_config, 'r') as file:
    yaml_content = file.read()
autoencoder_args = yaml.safe_load(yaml_content)

# Initialize encoder
encoder_model = EncoderModel(
    in_dim=adata.shape[1],
    n_cat=adata.obs[covariate_keys].unique().shape[0],
    conditioning_covariate=covariate_keys,
    encoder_type='learnt_autoencoder',
    **autoencoder_args
)

# Load weights 
encoder_model.load_state_dict(torch.load(ae_path)["state_dict"])

/tmp/ipykernel_695361/1251314608.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder_model.load_state_dict(torch.load(ae_path)["state_dict"])


<All keys matched successfully>

In [16]:
defaults = dict(
    clip_denoised=True,
    batch_size=64,
    sample_fn="ddim",
    multimodal_model_path="/home/wuboyang/scduo-new/script/training_diffusion/outputs/outputs6/train_outputs/model200000.pt",
    output_dir="test",
    classifier_scale=0,
    devices='0',
    is_strict=True,
    all_save_num= 1024,
    seed=42,
    load_noise="",
    data_dir=dataset_path,
    condition='cell_type',
)
defaults.update(model_and_diffusion_defaults())
parser = argparse.ArgumentParser()
defaults['ctrl_dim'] = '1,100'
defaults['pert_dim'] = '1,100'
defaults['num_channels'] = 128
defaults['num_res_blocks'] = 1
defaults['resblock_updown'] = True
defaults['num_class'] = 7
defaults['class_cond'] = True
add_dict_to_argparser(parser, defaults)

In [18]:
# load diffusion backbone
args = parser.parse_known_args()[0]
args.ctrl_dim = [int(i) for i in args.ctrl_dim.split(',')]
args.pert_dim = [int(i) for i in args.pert_dim.split(',')]

dist_util.setup_dist(args.devices)

print("creating model and diffusion...")
multimodal_model, multimodal_diffusion = create_model_and_diffusion(
        **args_to_dict(args, [key for key in model_and_diffusion_defaults().keys()])
)
multimodal_model.load_state_dict_(
        dist_util.load_state_dict(args.multimodal_model_path, map_location="cpu"), is_strict=args.is_strict
)
multimodal_model.to(dist_util.dev())
optimizer2 = optim.Adam(multimodal_model.parameters(), lr=0.001)

# mdata = mu.read_h5mu(args.data_dir)
from sklearn.preprocessing import LabelEncoder
labels = adata['pert'].obs[args.condition].values
label_encoder = LabelEncoder()
label_encoder.fit(labels)
classes_all = label_encoder.transform(labels)

ctrl = adata[adata.obs[args.condition] == "control"].copy()
pert = adata[adata.obs[args.condition] == "stimulated"].copy()

creating model and diffusion...
audio_gene_layers.0.weight not exists in state_dict
audio_gene_layers.0.bias not exists in state_dict
audio_gene_layers.2.weight not exists in state_dict
audio_gene_layers.2.bias not exists in state_dict


RuntimeError: Error(s) in loading state_dict for MultimodalUNet:
	Missing key(s) in state_dict: "audio_gene_layers.0.weight", "audio_gene_layers.0.bias", "audio_gene_layers.2.weight", "audio_gene_layers.2.bias". 

In [ ]:
""